In [2]:
!pip install scikit-learn

In [3]:
%pip install xgboost==1.7.6

Note: you may need to restart the kernel to use updated packages.


In [4]:
import xgboost as xgb
print("XGBoost version:", xgb.__version__)

XGBoost version: 1.7.6


In [5]:
# Setup of SageMaker session
import time
import boto3
import numpy as np
import pandas as pd

import sagemaker
from sagemaker import get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.inputs import OfflineStoreConfig, S3StorageConfig
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

session = sagemaker.Session()
region = session.boto_region_name
bucket = session.default_bucket()
role = get_execution_role()

featurestore_runtime = boto3.client(
    "sagemaker-featurestore-runtime",
    region_name=region
)

sagemaker_client = boto3.client(
    "sagemaker",
    region_name=region
)

print("Region:", region)
print("Bucket:", bucket)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Region: us-east-1
Bucket: sagemaker-us-east-1-451813790848


In [6]:
# Project Settings
LOCAL_DATA_PATH = "inference_modeling_dataset.csv"

PROJECT_PREFIX = "prj-inference-power-feature-store"
FEATURE_GROUP_NAME = f"inference-power-features-{int(time.time())}"

offline_store_s3_uri = f"s3://{bucket}/{PROJECT_PREFIX}/offline-store/"
training_s3_uri = f"s3://{bucket}/{PROJECT_PREFIX}/training/"
model_s3_uri = f"s3://{bucket}/{PROJECT_PREFIX}/model/"

print("Feature Group:", FEATURE_GROUP_NAME)


Feature Group: inference-power-features-1791131080


In [7]:
# Load the project dataset
df = pd.read_csv(LOCAL_DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())
display(df.dtypes)


Rows: 2424
Columns: 7


,workload_type,batch_size,max_output_tokens,mean_power_w,request_rate,output_length,dataset
0,offline_inference,550.0,1024.0,2822.104293,NaN,NaN,NaN
1,offline_inference,550.0,1024.0,2814.350863,NaN,NaN,NaN
2,offline_inference,550.0,1024.0,2798.014121,NaN,NaN,NaN
3,offline_inference,275.0,1024.0,2694.764568,NaN,NaN,NaN
4,offline_inference,275.0,1024.0,2696.992151,NaN,NaN,NaN


workload_type         object
batch_size           float64
max_output_tokens    float64
mean_power_w         float64
request_rate         float64
output_length        float64
dataset               object
dtype: object

In [8]:
# Before creating the Feature Group, we need to understand the workload types and the missing values. 
# Some missing values are expected because not every workload uses the same inputs.
print("Workload types:")
display(df["workload_type"].value_counts())

print("Missing values:")
display(df.isna().sum())

print("Target summary:")
display(df["mean_power_w"].describe())


Workload types:


workload_type
offline_inference          1200
online_finite_inference    1024
online_rate_inference       200
Name: count, dtype: int64

Missing values:


workload_type           0
batch_size           1224
max_output_tokens    1224
mean_power_w            0
request_rate         1200
output_length        1400
dataset              1200
dtype: int64

Target summary:


count    2424.000000
mean     2593.952042
std       184.353792
min      1955.359516
25%      2481.449575
50%      2601.223149
75%      2755.523082
max      2888.573990
Name: mean_power_w, dtype: float64

Feature engineering
The categorical columns need to be converted into numeric features.

We need to:
- one-hot encode `workload_type`
- one-hot encode `dataset`
- fill missing numeric inputs with 0
- keep `mean_power_w` as the target
- add a record ID and event time for Feature Store

We can use `dtype=int` for the one-hot columns so they are stored as 0/1 rather than Pandas Boolean values.

In [9]:
feature_df = df.copy()

# "dataset" is not applicable for some workload types
feature_df["dataset"] = feature_df["dataset"].fillna("not_applicable")

# Make the category names safe for Feature Store / Athena
feature_df["dataset"] = (
    feature_df["dataset"]
    .astype(str)
    .str.replace("/", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

# Convert categorical variables to 0/1 columns
feature_df = pd.get_dummies(
    feature_df,
    columns=["workload_type", "dataset"],
    dtype=int
)

# These fields are not available for every workload type
numeric_columns = [
    "batch_size",
    "max_output_tokens",
    "request_rate",
    "output_length"
]

for column in numeric_columns:
    feature_df[column] = feature_df[column].fillna(0)

# Feature Store metadata
feature_df["record_id"] = np.arange(len(feature_df)).astype(str)
feature_df["event_time"] = time.time()

target_column = "mean_power_w"

print("New shape:", feature_df.shape)
display(feature_df.head())


New shape: (2424, 13)


,batch_size,max_output_tokens,mean_power_w,request_rate,output_length,workload_type_offline_inference,workload_type_online_finite_inference,workload_type_online_rate_inference,dataset_likaixin_InstructCoder,dataset_mgoin_mlperf_inference_llama2_data,dataset_not_applicable,record_id,event_time
0,550.0,1024.0,2822.104293,0.0,0.0,1,0,0,0,0,1,0,1.791131e+09
1,550.0,1024.0,2814.350863,0.0,0.0,1,0,0,0,0,1,1,1.791131e+09
2,550.0,1024.0,2798.014121,0.0,0.0,1,0,0,0,0,1,2,1.791131e+09
3,275.0,1024.0,2694.764568,0.0,0.0,1,0,0,0,0,1,3,1.791131e+09
4,275.0,1024.0,2696.992151,0.0,0.0,1,0,0,0,0,1,4,1.791131e+09


In [10]:
# Check the transformed features
print("Feature columns:")
print(feature_df.columns.tolist())

print("\nData types:")
display(feature_df.dtypes)


Feature columns:
['batch_size', 'max_output_tokens', 'mean_power_w', 'request_rate', 'output_length', 'workload_type_offline_inference', 'workload_type_online_finite_inference', 'workload_type_online_rate_inference', 'dataset_likaixin_InstructCoder', 'dataset_mgoin_mlperf_inference_llama2_data', 'dataset_not_applicable', 'record_id', 'event_time']

Data types:


batch_size                                    float64
max_output_tokens                             float64
mean_power_w                                  float64
request_rate                                  float64
output_length                                 float64
workload_type_offline_inference                 int64
workload_type_online_finite_inference           int64
workload_type_online_rate_inference             int64
dataset_likaixin_InstructCoder                  int64
dataset_mgoin_mlperf_inference_llama2_data      int64
dataset_not_applicable                          int64
record_id                                      object
event_time                                    float64
dtype: object

In [11]:
# Create the Feature Group object
# This project has one modeling dataset, so one Feature Group is enough.
# The target is kept in the Feature Group because we will use the Offline Store to build the training dataset.
feature_columns = [
    column
    for column in feature_df.columns
    if column not in [target_column, "record_id", "event_time"]
]

feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME,
    sagemaker_session=session
)

feature_group.load_feature_definitions(
    data_frame=feature_df
)

print("Number of Feature Store columns:",
      len(feature_group.feature_definitions))


Number of Feature Store columns: 13


Create the Feature Grop
Both stores are enabled:

- Online Store - for low-latency feature lookup during inference
- Offline Store - for S3/Athena access during training

In [12]:
print("Feature Group:", FEATURE_GROUP_NAME)
print("Data shape:", feature_df.shape)
print("Columns:")
for c in feature_df.columns:
    print("  ", c)

feature_group.load_feature_definitions(
    data_frame=feature_df
)

print("\nFeature definitions:")
for fd in feature_group.feature_definitions:
    print(fd)

print("\nCount:", len(feature_group.feature_definitions))

Feature Group: inference-power-features-1791131080
Data shape: (2424, 13)
Columns:
   batch_size
   max_output_tokens
   mean_power_w
   request_rate
   output_length
   workload_type_offline_inference
   workload_type_online_finite_inference
   workload_type_online_rate_inference
   dataset_likaixin_InstructCoder
   dataset_mgoin_mlperf_inference_llama2_data
   dataset_not_applicable
   record_id
   event_time

Feature definitions:
FeatureDefinition(feature_name='batch_size', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None)
FeatureDefinition(feature_name='max_output_tokens', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None)
FeatureDefinition(feature_name='mean_power_w', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None)
FeatureDefinition(feature_name='request_rate', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None)
FeatureDefinition(feature_name='output_length', feature_

In [13]:
# Load feature definitions
feature_group.load_feature_definitions(data_frame=feature_df)

# Verify
print("Feature count:", len(feature_group.feature_definitions))
print("Columns:", feature_df.columns.tolist())

# Create Feature Group
feature_group.create(
    s3_uri=offline_store_s3_uri,
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True
)

print("Feature Group creation requested.")


Feature count: 13
Columns: ['batch_size', 'max_output_tokens', 'mean_power_w', 'request_rate', 'output_length', 'workload_type_offline_inference', 'workload_type_online_finite_inference', 'workload_type_online_rate_inference', 'dataset_likaixin_InstructCoder', 'dataset_mgoin_mlperf_inference_llama2_data', 'dataset_not_applicable', 'record_id', 'event_time']
Feature Group creation requested.


Wait for the Feature Group to be created

In [14]:
while True:
    response = sagemaker_client.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )

    status = response["FeatureGroupStatus"]
    print("Status:", status)

    if status == "Created":
        break

    if status in ["CreateFailed", "DeleteFailed"]:
        raise RuntimeError(
            response.get("FailureReason", "Feature Group creation failed.")
        )

    time.sleep(5)


Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Created


Load the data into Feature Store

In [15]:
feature_group.ingest(
    data_frame=feature_df,
    max_workers=3,
    wait=True
)

print("Feature ingestion completed.")


Feature ingestion completed.


In [25]:
print("Feature Group status:")
print(feature_group.describe()["FeatureGroupStatus"])

print("\nLast ingestion:")
print(feature_group.describe().get("LastProcessingResult"))

print("\nRecord count in source:")
print(len(feature_df))

Feature Group status:
Created

Last ingestion:
None

Record count in source:
2424


In [26]:
import boto3

sm = boto3.client("sagemaker")

desc = sm.describe_feature_group(
    FeatureGroupName=FEATURE_GROUP_NAME
)

print("Status:", desc["FeatureGroupStatus"])
print("Offline store:", desc.get("OfflineStoreStatus"))

Status: Created
Offline store: {'Status': 'Active'}


Check the Online Store
Retrieve one record using the `record_id` that was created above.

In [27]:
sample_record_id = feature_df.iloc[0]["record_id"]

response = featurestore_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    RecordIdentifierValueAsString=sample_record_id
)

response


{'ResponseMetadata': {'RequestId': '61eeba9c-b5f0-4e18-8f18-dc14782ac664',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '61eeba9c-b5f0-4e18-8f18-dc14782ac664',
   'content-type': 'application/json',
   'content-length': '1188',
   'date': 'Sun, 04 Oct 2026 16:40:08 GMT'},
  'RetryAttempts': 0},
 'Record': [{'FeatureName': 'batch_size', 'ValueAsString': '550.0'},
  {'FeatureName': 'max_output_tokens', 'ValueAsString': '1024.0'},
  {'FeatureName': 'mean_power_w', 'ValueAsString': '2822.1042929558016'},
  {'FeatureName': 'request_rate', 'ValueAsString': '0.0'},
  {'FeatureName': 'output_length', 'ValueAsString': '0.0'},
  {'FeatureName': 'workload_type_offline_inference', 'ValueAsString': '1'},
  {'FeatureName': 'workload_type_online_finite_inference',
   'ValueAsString': '0'},
  {'FeatureName': 'workload_type_online_rate_inference', 'ValueAsString': '0'},
  {'FeatureName': 'dataset_likaixin_InstructCoder', 'ValueAsString': '0'},
  {'FeatureName': 'dataset_mgoin_mlperf_i

Read the Online record

In [28]:
online_record = {
    item["FeatureName"]: item["ValueAsString"]
    for item in response["Record"]
}

display(pd.DataFrame([online_record]))


,batch_size,max_output_tokens,mean_power_w,request_rate,output_length,workload_type_offline_inference,workload_type_online_finite_inference,workload_type_online_rate_inference,dataset_likaixin_InstructCoder,dataset_mgoin_mlperf_inference_llama2_data,dataset_not_applicable,record_id,event_time
0,550.0,1024.0,2822.1042929558016,0.0,0.0,1,0,0,0,0,1,0,1791131148.4398694


Check the Offline Store for records

In [29]:
athena_query = feature_group.athena_query()
table_name = athena_query.table_name

query = f'''
SELECT COUNT(*) AS record_count
FROM "{table_name}"
'''

athena_query.run(
    query_string=query,
    output_location=f"s3://{bucket}/{PROJECT_PREFIX}/athena-results/"
)

athena_query.wait()

count_df = athena_query.as_dataframe()
display(count_df)

,record_count
0,2424


Check the Offline Store through Athena
The Offline Store is populated asynchronously. The Feature Store Athena query helper gives us the table name created for the Feature Group.

In [30]:
athena_query = feature_group.athena_query()
table_name = athena_query.table_name

print("Athena table:", table_name)

query = f'''
SELECT *
FROM "{table_name}"
LIMIT 5
'''

athena_query.run(
    query_string=query,
    output_location=f"s3://{bucket}/{PROJECT_PREFIX}/athena-results/"
)

athena_query.wait()

offline_sample = athena_query.as_dataframe()
display(offline_sample)


Athena table: inference_power_features_1791131080_1791131373


,batch_size,max_output_tokens,mean_power_w,request_rate,output_length,workload_type_offline_inference,workload_type_online_finite_inference,workload_type_online_rate_inference,dataset_likaixin_instructcoder,dataset_mgoin_mlperf_inference_llama2_data,dataset_not_applicable,record_id,event_time,write_time,api_invocation_time,is_deleted
0,325.0,512.0,2789.494157,0.0,0.0,1,0,0,0,0,1,809,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
1,850.0,512.0,2698.900436,0.0,0.0,1,0,0,0,0,1,31,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
2,0.0,0.0,2244.238600,10.0,256.0,0,1,0,1,0,0,1656,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
3,0.0,0.0,2574.402776,700.0,256.0,0,1,0,0,1,0,1690,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
4,375.0,512.0,2350.152476,0.0,0.0,1,0,0,0,0,1,80,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False


In [31]:
print("Shape:", offline_sample.shape)
print("\nColumns:")
for c in offline_sample.columns:
    print(c)

display(offline_sample.head())

Shape: (5, 16)

Columns:
batch_size
max_output_tokens
mean_power_w
request_rate
output_length
workload_type_offline_inference
workload_type_online_finite_inference
workload_type_online_rate_inference
dataset_likaixin_instructcoder
dataset_mgoin_mlperf_inference_llama2_data
dataset_not_applicable
record_id
event_time
write_time
api_invocation_time
is_deleted


,batch_size,max_output_tokens,mean_power_w,request_rate,output_length,workload_type_offline_inference,workload_type_online_finite_inference,workload_type_online_rate_inference,dataset_likaixin_instructcoder,dataset_mgoin_mlperf_inference_llama2_data,dataset_not_applicable,record_id,event_time,write_time,api_invocation_time,is_deleted
0,325.0,512.0,2789.494157,0.0,0.0,1,0,0,0,0,1,809,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
1,850.0,512.0,2698.900436,0.0,0.0,1,0,0,0,0,1,31,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
2,0.0,0.0,2244.238600,10.0,256.0,0,1,0,1,0,0,1656,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
3,0.0,0.0,2574.402776,700.0,256.0,0,1,0,0,1,0,1690,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False
4,375.0,512.0,2350.152476,0.0,0.0,1,0,0,0,0,1,80,1.791131e+09,2026-10-04 16:35:23.312,2026-10-04 16:30:07.000,False


In [32]:
print("mean_power_w nulls:", offline_sample["mean_power_w"].isna().sum())
print("record_id nulls:", offline_sample["record_id"].isna().sum())
print("event_time nulls:", offline_sample["event_time"].isna().sum())

mean_power_w nulls: 0
record_id nulls: 0
event_time nulls: 0


Build the training dataset
`record_id` and `event_time` are Feature Store metadata, so they are not used as model features.

In [33]:
training_columns = [target_column] + feature_columns

select_columns = ", ".join(
    [f'"{column}"' for column in training_columns]
)

query = f'''
SELECT {select_columns}
FROM "{table_name}"
WHERE "{target_column}" IS NOT NULL
'''

athena_query.run(
    query_string=query,
    output_location=f"s3://{bucket}/{PROJECT_PREFIX}/athena-results/"
)

athena_query.wait()

training_df = athena_query.as_dataframe()

print("Training data shape:", training_df.shape)
display(training_df.head())


Training data shape: (2424, 11)


,mean_power_w,batch_size,max_output_tokens,request_rate,output_length,workload_type_offline_inference,workload_type_online_finite_inference,workload_type_online_rate_inference,dataset_likaixin_InstructCoder,dataset_mgoin_mlperf_inference_llama2_data,dataset_not_applicable
0,2522.479982,0.0,0.0,40.0,1024.0,0,1,0,0,1,0
1,2802.089283,450.0,512.0,0.0,0.0,1,0,0,0,0,1
2,2643.141153,875.0,512.0,0.0,0.0,1,0,0,0,0,1
3,2687.457250,750.0,512.0,0.0,0.0,1,0,0,0,0,1
4,2777.138064,475.0,512.0,0.0,0.0,1,0,0,0,0,1


Train / Validation Split

In [34]:
X = training_df[feature_columns].astype(float)
y = training_df[target_column].astype(float)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))


Training rows: 1939
Validation rows: 485


Upload the training data to S3

In [35]:
train_df = pd.concat(
    [
        y_train.reset_index(drop=True),
        X_train.reset_index(drop=True)
    ],
    axis=1
)

valid_df = pd.concat(
    [
        y_valid.reset_index(drop=True),
        X_valid.reset_index(drop=True)
    ],
    axis=1
)

train_local = "/tmp/train.csv"
valid_local = "/tmp/validation.csv"

train_df.to_csv(train_local, header=False, index=False)
valid_df.to_csv(valid_local, header=False, index=False)

s3_train = session.upload_data(
    path=train_local,
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/training/train"
)

s3_validation = session.upload_data(
    path=valid_local,
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/training/validation"
)

print("Training data:", s3_train)
print("Validation data:", s3_validation)


Training data: s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/training/train/train.csv
Validation data: s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/training/validation/validation.csv


Five-model strategy
The project evaluates the five required regression approaches using SageMaker:

1. Baseline 0 — Global Mean
2. Baseline 1 — Workload Mean
3. Linear Regression
4. Random Forest
5. XGBoost

The baselines and the scikit-learn models use SageMaker's managed SKLearn Estimator with small training scripts written below. XGBoost uses the SageMaker built-in XGBoost container.

All five models use the same training/validation data and the same validation metrics so the comparison is fair.


Create SageMaker training scripts
The two baselines and the Linear Regression / Random Forest models are packaged as small Python training scripts and executed by SageMaker's managed scikit-learn container.

This keeps the modeling logic in SageMaker rather than calculating the models locally.

In [36]:
from pathlib import Path

script_dir = Path("/tmp/sagemaker_scripts")
script_dir.mkdir(exist_ok=True)

# Baseline 0: global mean
(script_dir / "baseline_global_mean.py").write_text(r"""
import argparse, os, glob, json
import pandas as pd
import joblib

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--target", type=str, default="mean_power_w")
    args = parser.parse_args()

    train_files = glob.glob("/opt/ml/input/data/train/*.csv")
    train_path = train_files[0]
    df = pd.read_csv(train_path, header=None)

    y = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    mean_value = float(y.mean())

    model = {"type": "global_mean", "mean_power_w": mean_value}
    os.makedirs("/opt/ml/model", exist_ok=True)
    joblib.dump(model, "/opt/ml/model/model.joblib")

if __name__ == "__main__":
    main()
""")

# Baseline 1: workload mean.
# Because the Feature Store data is one-hot encoded, infer workload columns from names.
(script_dir / "baseline_workload_mean.py").write_text(r"""
import argparse, os, glob
import pandas as pd
import joblib

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--workload-columns", type=str, required=True)
    args = parser.parse_args()

    train_path = glob.glob("/opt/ml/input/data/train/*.csv")[0]
    df = pd.read_csv(train_path, header=None)

    workload_cols = [x for x in args.workload_columns.split(",") if x]
    # Feature order in this notebook is: target + feature_columns.
    # Workload columns are identified by their zero-based feature positions
    # supplied at training-job creation time.
    y = pd.to_numeric(df.iloc[:, 0], errors="coerce")

    means = {}
    for col_name, idx_text in [x.split("=") for x in args.workload_columns.split(",")]:
        idx = int(idx_text)
        mask = pd.to_numeric(df.iloc[:, idx], errors="coerce") == 1
        if mask.any():
            means[col_name] = float(y[mask].mean())

    global_mean = float(y.mean())
    model = {
        "type": "workload_mean",
        "means": means,
        "global_mean": global_mean
    }

    os.makedirs("/opt/ml/model", exist_ok=True)
    joblib.dump(model, "/opt/ml/model/model.joblib")

if __name__ == "__main__":
    main()
""")

# Generic sklearn trainer: linear regression or random forest.
(script_dir / "sklearn_regression.py").write_text(r"""
import argparse, os, glob
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-type", type=str, required=True)
    parser.add_argument("--n-estimators", type=int, default=300)
    parser.add_argument("--max-depth", type=int, default=None)
    parser.add_argument("--random-state", type=int, default=42)
    args = parser.parse_args()

    train_path = glob.glob("/opt/ml/input/data/train/*.csv")[0]
    df = pd.read_csv(train_path, header=None)

    y = pd.to_numeric(df.iloc[:, 0], errors="coerce").astype(float)
    X = df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").astype(float)

    if args.model_type == "linear":
        model = LinearRegression()
    elif args.model_type == "random_forest":
        model = RandomForestRegressor(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth,
            random_state=args.random_state,
            n_jobs=-1
        )
    else:
        raise ValueError("Unsupported model type")

    model.fit(X, y)

    os.makedirs("/opt/ml/model", exist_ok=True)
    joblib.dump(model, "/opt/ml/model/model.joblib")

if __name__ == "__main__":
    main()
""")

print("Training scripts written to:", script_dir)


Training scripts written to: /tmp/sagemaker_scripts


Common SageMaker SKLearn estimator configuration
The following helper creates a managed SageMaker training job for the baseline/Linear/RF models.

The training data is already uploaded to S3

In [37]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

sklearn_framework_version = "1.4-2"
sklearn_py_version = "py3"
sklearn_instance_type = "ml.m5.large"

def make_sklearn_estimator(entry_point, job_name, hyperparameters=None):
    return SKLearn(
        entry_point=entry_point,
        source_dir=str(script_dir),
        role=role,
        framework_version=sklearn_framework_version,
        py_version=sklearn_py_version,
        instance_type=sklearn_instance_type,
        instance_count=1,
        output_path=f"{model_s3_uri}{job_name}/",
        sagemaker_session=session,
        hyperparameters=hyperparameters or {},
        base_job_name=job_name
    )

train_input = TrainingInput(s3_train, content_type="text/csv")
validation_input = TrainingInput(s3_validation, content_type="text/csv")

print("SageMaker training inputs:")
print("Train:", s3_train)
print("Validation:", s3_validation)


SageMaker training inputs:
Train: s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/training/train/train.csv
Validation: s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/training/validation/validation.csv


Baseline 0 - Global Mean
This is the simplest possible model.
It calculates the mean `mean_power_w` from the training set and predicts that same value for every validation observation.
The calculation is performed inside a SageMaker training job and the resulting baseline artifact is stored in the job's S3 output.

In [39]:
global_estimator = make_sklearn_estimator(
    entry_point="baseline_global_mean.py",
    job_name="baseline-global-mean"
)

global_estimator.fit({
    "train": train_input
})

print("Global Mean training completed.")
print("Global Mean:", global_estimator.model_data)

INFO:sagemaker:Creating training-job with name: baseline-global-mean-2026-10-04-16-45-47-466


2026-10-04 16:45:48 Starting - Starting the training job...
2026-10-04 16:46:03 Starting - Preparing the instances for training...
2026-10-04 16:46:30 Downloading - Downloading input data...
2026-10-04 16:47:20 Training - Training image download completed. Training in progress..2026-10-04 16:47:21,508 sagemaker-training-toolkit INFO     Imported framework sagemaker_sklearn_container.training
2026-10-04 16:47:21,509 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 16:47:21,509 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 16:47:21,519 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-10-04 16:47:21,736 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 16:47:21,736 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 16:47:21,747 sagemaker-training-toolkit INFO     No G

Baseline 1 - Workload Mean
This baseline predicts a different mean for each workload type.

The current Feature Store representation one-hot encodes workload type. The script therefore uses the one-hot workload columns to calculate the training-set mean for each workload category.

If a workload category is not available for a new record, the global training mean is used as the fallback.

In [40]:
# Identify one-hot workload columns in the same feature order used above.
workload_feature_names = [
    c for c in feature_columns
    if c.startswith("workload_type_")
]

# The CSV passed to SageMaker has target at column 0 and feature columns after it.
# Therefore feature index in the CSV = feature_columns.index(name) + 1.
workload_arg = ",".join(
    f"{name}={feature_columns.index(name)+1}"
    for name in workload_feature_names
)

print("Workload columns passed to baseline:", workload_arg)

workload_estimator = make_sklearn_estimator(
    entry_point="baseline_workload_mean.py",
    job_name="baseline-workload-mean",
    hyperparameters={"workload-columns": workload_arg}
)

workload_estimator.fit({
    "train": train_input
})

print("Baseline 1 training completed.")
print("Workload Mean:", workload_estimator.model_data)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Workload columns passed to baseline: workload_type_offline_inference=5,workload_type_online_finite_inference=6,workload_type_online_rate_inference=7


INFO:sagemaker:Creating training-job with name: baseline-workload-mean-2026-10-04-16-49-25-132


2026-10-04 16:49:26 Starting - Starting the training job...
2026-10-04 16:49:42 Starting - Preparing the instances for training...
2026-10-04 16:50:07 Downloading - Downloading input data...
2026-10-04 16:50:58 Training - Training image download completed. Training in progress..2026-10-04 16:50:59,300 sagemaker-training-toolkit INFO     Imported framework sagemaker_sklearn_container.training
2026-10-04 16:50:59,301 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 16:50:59,302 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 16:50:59,312 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-10-04 16:50:59,528 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 16:50:59,529 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 16:50:59,541 sagemaker-training-toolkit INFO     No G

Linear Regression - SageMaker
Linear Regression tests whether the configuration variables explain mean power approximately through a linear relationship.
The model is trained in a SageMaker-managed scikit-learn container.

In [42]:
linear_estimator = make_sklearn_estimator(
    entry_point="sklearn_regression.py",
    job_name="linear-regression",
    hyperparameters={
        "model-type": "linear",
        "random-state": 42
    }
)

linear_estimator.fit({
    "train": train_input
})

print("Linear Regression training completed.")
print("Linear Regression:", linear_estimator.model_data)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: linear-regression-2026-10-04-16-58-35-200


2026-10-04 16:58:36 Starting - Starting the training job...
2026-10-04 16:58:50 Starting - Preparing the instances for training...
2026-10-04 16:59:17 Downloading - Downloading input data...
2026-10-04 17:00:07 Downloading - Downloading the training image..2026-10-04 17:00:14,320 sagemaker-training-toolkit INFO     Imported framework sagemaker_sklearn_container.training
2026-10-04 17:00:14,321 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 17:00:14,322 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 17:00:14,333 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-10-04 17:00:14,557 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 17:00:14,557 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 17:00:14,569 sagemaker-training-toolkit INFO     No GPUs detected (normal i

Random Forest - SageMaker, 
Random Forest captures nonlinear relationships and interactions among workload configuration variables.
The model is trained in a SageMaker-managed scikit-learn container.

In [43]:
rf_estimator = make_sklearn_estimator(
    entry_point="sklearn_regression.py",
    job_name="random-forest",
    hyperparameters={
        "model-type": "random_forest",
        "n-estimators": 300,
        "max-depth": 12,
        "random-state": 42
    }
)

rf_estimator.fit({
    "train": train_input
})

print("Random Forest training completed.")
print("Random Forest:", rf_estimator.model_data)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: random-forest-2026-10-04-17-01-14-586


2026-10-04 17:01:19 Starting - Starting the training job...
2026-10-04 17:01:34 Starting - Preparing the instances for training...
2026-10-04 17:02:07 Downloading - Downloading input data.....2026-10-04 17:03:01,879 sagemaker-training-toolkit INFO     Imported framework sagemaker_sklearn_container.training
2026-10-04 17:03:01,880 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 17:03:01,881 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 17:03:01,891 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-10-04 17:03:02,147 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 17:03:02,147 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-10-04 17:03:02,159 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-10-04 17:03:02,160 sagemaker-training-t

XGBoost SageMaker - Built-in Algorithm
This follows the SageMaker XGBoost approach from the original notebook.
The model is trained using SageMaker's managed XGBoost container.

In [41]:
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve

xgb_image = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.m5.large"
)

xgb_estimator = Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"{model_s3_uri}xgboost/",
    sagemaker_session=session,
    base_job_name="xgboost-power"
)

xgb_estimator.set_hyperparameters(
    objective="reg:squarederror",
    num_round=100,
    max_depth=5,
    eta=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="rmse"
)

xgb_estimator.fit({
    "train": train_input,
    "validation": validation_input
})

print("XGBoost training completed.")
print("XGBoost:", xgb_estimator.model_data)

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via Telemet

2026-10-04 16:53:37 Starting - Starting the training job...
2026-10-04 16:53:52 Starting - Preparing the instances for training...
2026-10-04 16:54:18 Downloading - Downloading input data.........
2026-10-04 16:56:09 Training - Training image download completed. Training in progress.
2026-10-04 16:56:09 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-10-04 16:56:03.766 ip-10-0-116-99.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-10-04 16:56:03.829 ip-10-0-116-99.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-10-04:16:56:04:INFO] Imported framework sagemaker_xgboost_container.traini

In [44]:
import os
import tarfile
import boto3

s3 = boto3.client("s3")

model_artifacts = {
    "Global Mean": global_estimator.model_data,
    "Workload Mean": workload_estimator.model_data,
    "Linear Regression": linear_estimator.model_data,
    "Random Forest": rf_estimator.model_data,
    "XGBoost": xgb_estimator.model_data,
}

artifact_dirs = {}

for model_name, s3_uri in model_artifacts.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    # Parse S3 URI
    s3_parts = s3_uri.replace("s3://", "").split("/", 1)
    bucket_name = s3_parts[0]
    key = s3_parts[1]

    # Local paths
    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
    )

    local_tar = f"/tmp/{safe_name}.tar.gz"
    extract_dir = f"/tmp/{safe_name}_artifact"

    os.makedirs(extract_dir, exist_ok=True)

    # Download
    print("Downloading:")
    print(s3_uri)

    s3.download_file(
        bucket_name,
        key,
        local_tar
    )

    # Extract
    with tarfile.open(
        local_tar,
        "r:gz"
    ) as tar:

        tar.extractall(
            path=extract_dir
        )

    artifact_dirs[model_name] = extract_dir

    # Show contents
    print("\nArtifact contents:")

    for root, dirs, files in os.walk(extract_dir):

        for file in files:

            full_path = os.path.join(
                root,
                file
            )

            print(
                "  ",
                os.path.relpath(
                    full_path,
                    extract_dir
                )
            )


Global Mean
Downloading:
s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/model/baseline-global-mean/baseline-global-mean-2026-10-04-16-45-47-466/output/model.tar.gz

Artifact contents:
   model.joblib

Workload Mean
Downloading:
s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/model/baseline-workload-mean/baseline-workload-mean-2026-10-04-16-49-25-132/output/model.tar.gz

Artifact contents:
   model.joblib

Linear Regression
Downloading:
s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/model/linear-regression/linear-regression-2026-10-04-16-58-35-200/output/model.tar.gz

Artifact contents:
   model.joblib

Random Forest
Downloading:
s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/model/random-forest/random-forest-2026-10-04-17-01-14-586/output/model.tar.gz

Artifact contents:
   model.joblib

XGBoost
Downloading:
s3://sagemaker-us-east-1-451813790848/prj-inference-power-feature-store/model/xg

In [45]:
import joblib
import os
import xgboost as xgb

# --------------------------------------------------
# Locate extracted artifacts from previous cell
# --------------------------------------------------

global_model_path = os.path.join(
    artifact_dirs["Global Mean"],
    "model.joblib"
)

workload_model_path = os.path.join(
    artifact_dirs["Workload Mean"],
    "model.joblib"
)

linear_model_path = os.path.join(
    artifact_dirs["Linear Regression"],
    "model.joblib"
)

rf_model_path = os.path.join(
    artifact_dirs["Random Forest"],
    "model.joblib"
)

xgb_model_path = os.path.join(
    artifact_dirs["XGBoost"],
    "xgboost-model"
)


# --------------------------------------------------
# Load models
# --------------------------------------------------

global_model = joblib.load(global_model_path)

workload_model = joblib.load(workload_model_path)

linear_model = joblib.load(linear_model_path)

rf_model = joblib.load(rf_model_path)

xgb_model = xgb.Booster()

xgb_model.load_model(xgb_model_path)


# --------------------------------------------------
# Confirm
# --------------------------------------------------

print("Global Mean model:")
print(global_model)

print("\nWorkload Mean model:")
print(workload_model)

print("\nLinear Regression model:")
print(type(linear_model))

print("\nRandom Forest model:")
print(type(rf_model))

print("\nXGBoost model:")
print(type(xgb_model))

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Global Mean model:
{'type': 'global_mean', 'mean_power_w': 2595.671034802987}

Workload Mean model:
{'type': 'workload_mean', 'means': {'workload_type_offline_inference': 2667.1605490881825, 'workload_type_online_finite_inference': 2492.241366745276, 'workload_type_online_rate_inference': 2681.504141629924}, 'global_mean': 2595.671034802987}

Linear Regression model:
<class 'sklearn.linear_model._base.LinearRegression'>

Random Forest model:
<class 'sklearn.ensemble._forest.RandomForestRegressor'>

XGBoost model:
<class 'xgboost.core.Booster'>


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.4.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [46]:
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

print("\nValidation columns:")
print(X_valid.columns.tolist())

print("\nLinear model features:")
print(linear_model.n_features_in_)

X_valid shape: (485, 10)
y_valid shape: (485,)

Validation columns:
['batch_size', 'max_output_tokens', 'request_rate', 'output_length', 'workload_type_offline_inference', 'workload_type_online_finite_inference', 'workload_type_online_rate_inference', 'dataset_likaixin_InstructCoder', 'dataset_mgoin_mlperf_inference_llama2_data', 'dataset_not_applicable']

Linear model features:
10


In [47]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------------------------------------------------
# 1. Generate predictions
# ---------------------------------------------------------

# Global Mean
global_pred = np.full(
    len(X_valid),
    global_model["mean_power_w"]
)

# Workload Mean
workload_pred = []

for _, row in X_valid.iterrows():
    if row["workload_type_offline_inference"] == 1:
        pred = workload_model["means"]["workload_type_offline_inference"]
    elif row["workload_type_online_finite_inference"] == 1:
        pred = workload_model["means"]["workload_type_online_finite_inference"]
    elif row["workload_type_online_rate_inference"] == 1:
        pred = workload_model["means"]["workload_type_online_rate_inference"]
    else:
        pred = workload_model["global_mean"]

    workload_pred.append(pred)

workload_pred = np.array(workload_pred)

# Linear Regression
linear_pred = linear_model.predict(X_valid)

# Random Forest
rf_pred = rf_model.predict(X_valid)

print("Predictions generated successfully.")
print("Global Mean:", len(global_pred))
print("Workload Mean:", len(workload_pred))
print("Linear Regression:", len(linear_pred))
print("Random Forest:", len(rf_pred))

Predictions generated successfully.
Global Mean: 485
Workload Mean: 485
Linear Regression: 485
Random Forest: 485


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [48]:
# ---------------------------------------------------------
# 2. Evaluate the four models
# ---------------------------------------------------------

def calculate_metrics(y_true, y_pred):
    return {
        "RMSE_W": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE_W": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

validation_results = []

predictions = {
    "Global Mean": global_pred,
    "Workload Mean": workload_pred,
    "Linear Regression": linear_pred,
    "Random Forest": rf_pred
}

for model_name, pred in predictions.items():
    metrics = calculate_metrics(y_valid, pred)

    validation_results.append({
        "Model": model_name,
        "RMSE_W": metrics["RMSE_W"],
        "MAE_W": metrics["MAE_W"],
        "R2": metrics["R2"]
    })

validation_results_df = pd.DataFrame(validation_results)

validation_results_df = validation_results_df.sort_values(
    by="RMSE_W",
    ascending=True
).reset_index(drop=True)

display(validation_results_df)

,Model,RMSE_W,MAE_W,R2
0,Random Forest,36.013206,25.979188,0.962174
1,Linear Regression,140.699021,101.337742,0.422635
2,Workload Mean,162.603789,111.471571,0.228867
3,Global Mean,185.367218,151.266926,-0.002153


In [49]:
import xgboost as xgb

dvalid = xgb.DMatrix(X_valid)

xgb_pred = xgb_model.predict(dvalid)

print("XGBoost predictions:", len(xgb_pred))

XGBoost predictions: 485


In [50]:
xgb_metrics = calculate_metrics(y_valid, xgb_pred)

validation_results.append({
    "Model": "XGBoost",
    "RMSE_W": xgb_metrics["RMSE_W"],
    "MAE_W": xgb_metrics["MAE_W"],
    "R2": xgb_metrics["R2"]
})

validation_results_df = pd.DataFrame(validation_results)

validation_results_df = validation_results_df.sort_values(
    by="RMSE_W",
    ascending=True
).reset_index(drop=True)

display(validation_results_df)

,Model,RMSE_W,MAE_W,R2
0,XGBoost,34.928412,25.386479,0.964418
1,Random Forest,36.013206,25.979188,0.962174
2,Linear Regression,140.699021,101.337742,0.422635
3,Workload Mean,162.603789,111.471571,0.228867
4,Global Mean,185.367218,151.266926,-0.002153
